# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset is described by its Croissant schema at:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("Published:", metadata.datePublished)
print("Keywords:", metadata.keywords)

## 2. Data Overview
Review available record sets and their details, referencing `@id` fields. Each RecordSet represents a distinct table or entity in the dataset.

In [ ]:
# List record sets (tables) and their fields, by `@id`
record_sets = list(dataset.record_sets())
print("Available RecordSets (with @id):\n")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '')}")
    # Show fields in this record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for fld in fields:
        if isinstance(fld, dict):
            fid, fname = fld.get('@id'), fld.get('name', '')
        else:
            fid, fname = fld, ''
        print(f"    - @id: {fid} {f'| name: {fname}' if fname else ''}")

## 3. Data Extraction
Load every RecordSet using its `@id` and display a preview. Always reference by `@id`.

In [ ]:
# Extract data for each RecordSet and store as DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"RecordSet @{rs_id} columns: {df.columns.tolist()}")
    print(df.head(2),'\n')

## 4. Exploratory Data Analysis (EDA)
Process the survey table by applying filtering, normalization, and grouping steps using columns referenced by their `@id`. RecordSet and Field/Column `@id`s are required throughout.

In [ ]:
# For demonstration, select the first RecordSet
main_rs_id = record_set_ids[0]
df = dataframes[main_rs_id]

# Identify a numeric field by @id (e.g., field for GAD-7 total score: find by printing column list if unsure)
print("Columns: ", df.columns.tolist())

# Let's choose the GAD-7 score column if present (replace with correct @id from overview step)
numeric_field_id = None
for col in df.columns:
    if 'gad' in col.lower() or 'total' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("Could not find a GAD-7 score field. Please update the code referencing the field @id.")
else:
    # Filter for high GAD-7 total scores (>10), normalize, and group by another field (e.g., gender or education), all referenced by @id
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Find 'gender' or 'education' field by @id for grouping
    group_field_id = None
    for col in df.columns:
        if ('gender' in col.lower()) or ('education' in col.lower()):
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Average {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df)

## 5. Visualization
Visualize the distributions of numeric assessment fields (e.g., GAD-7, PHQ-9, PSQ) and, if available, relationships to gender or education (by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If grouping field exists:
    if group_field_id:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook guided you through loading, exploring, and performing basic analyses on the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. For more advanced analysis (e.g., cross-record-set joins, machine learning), build upon these structured `@id`-based dataframes for rigorous, reproducible science.